## Set up
### Load Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import datetime as dt
from zoneinfo import ZoneInfo

### Load Raw Data
Cell takes ~5 minutes to run.

In [ ]:
df = pd.read_csv("~/Downloads/tnp.csv")

/var/folders/x2/60xrt5l55sb6kdqz4sc2_qnr0000gn/T/ipykernel_46999/3367628195.py:2: DtypeWarning: Columns (0: Shared Trip Authorized, 1: Shared Trip Match) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"../{filename}.csv")


### Cleaning and Processing

In [21]:
# Select relevant columns
data = df[
    [
        "Trip ID",
        "Trip Start Timestamp",
        "Trip End Timestamp",
        "Trip Seconds",
        "Trip Miles",
        "Percent Time Chicago",
        "Percent Distance Chicago",
        "Pickup Census Tract",
        "Dropoff Census Tract",
        "Fare",
        "Tip",
        "Additional Charges",
        "Trip Total",
        "Pickup Centroid Latitude",
        "Pickup Centroid Longitude",
        "Dropoff Centroid Latitude",
        "Dropoff Centroid Longitude",
    ]
]

# Drop all NAs
data = data.replace("NaN", np.nan)
data = data.dropna()

# Filter to only include trips 100% in Chicago
data["Int Percent Time Chicago"] = (
    data["Percent Time Chicago"].str.replace("%", "").str.replace(",", "")
).astype(int)

data["Int Percent Distance Chicago"] = (
    data["Percent Distance Chicago"].str.replace("%", "").str.replace(",", "")
).astype(int)

data = data[
    (data["Int Percent Time Chicago"] == 100)
    & (data["Int Percent Distance Chicago"] == 100)
]

# Convert timestamp fields to datetime format, specify Chicago time zone,
# convert to UTC (correct time zone for API).
data["Trip Start Timestamp"] = (
    pd.to_datetime(data["Trip Start Timestamp"], format="%m/%d/%Y %I:%M:%S %p")
    .dt.tz_localize(
        "America/Chicago",
        # if timestamp is ambiguous due to the daylight savings time
        # change, drop from dataset
        ambiguous="NaT",
        nonexistent="NaT",
    )
    .dt.tz_convert("UTC")
)
data["Trip End Timestamp"] = (
    pd.to_datetime(data["Trip End Timestamp"], format="%m/%d/%Y %I:%M:%S %p")
    .dt.tz_localize("America/Chicago", ambiguous="NaT", nonexistent="NaT")
    .dt.tz_convert("UTC")
)

data = data.dropna(subset=["Trip Start Timestamp", "Trip End Timestamp"])

# Specify boundaries in UTC
start_ct = pd.Timestamp("2025-01-01 00:00:00", tz="America/Chicago")
end_ct = pd.Timestamp("2025-12-31 23:59:59", tz="America/Chicago")

start_utc = start_ct.tz_convert("UTC")
end_utc = end_ct.tz_convert("UTC")

data = data[
    (data["Trip Start Timestamp"] >= start_utc)
    & (data["Trip End Timestamp"] <= end_utc)
]


print(f"Cleaned data has {len(data)} rows")

Cleaned data has 13007182 rows


## Grouping by Unique API Call
We have a limited number of API calls and want to use them efficiently. Rather than making repeated calls that return identical results, we will consolidate requests whenever possible.

Under our working assumptions--including that CTA schedules remain constant over the sample period (and match current schedule), that we ignore traffic variation, and that all weekdays are treated as equivalent—we will make one API call per unique set of parameters and apply the returned transit information to all rideshare trips that share those parameters.

Specifically, if multiple rideshare trips share the same:
- Start and end coordinates (rounded to three decimal places)
- Day type (0 = weekday, 1 = Saturday, 2 = Sunday)
- Start hour
then a single API call will suffice for all of them.

To implement this, we will:
1. Create a dataframe that counts the number of trips in each group defined by the criteria above.
2. Take a weighted random sample from this grouped dataframe.
3. Pass that sample to the API program.

This approach allows us to maximize sample size from the rideshare dataset (which is population data) while minimizing API usage.

### Grouping data

In [26]:
# round coodinates to 3 decimal points (~111m)
cols = [
    "Pickup Centroid Latitude",
    "Pickup Centroid Longitude",
    "Dropoff Centroid Latitude",
    "Dropoff Centroid Longitude",
]

data[cols] = data[cols].round(3)

# extract start hour
data["start_hour"] = data["Trip Start Timestamp"].dt.hour

# create day type variable
wd = data["Trip Start Timestamp"].dt.weekday
data["day_type"] = (wd == 5).astype(int) + (wd == 6).astype(int) * 2

# group on selected columns
group_cols = [
    "day_type",
    "start_hour",
    "Pickup Centroid Latitude",
    "Pickup Centroid Longitude",
    "Dropoff Centroid Latitude",
    "Dropoff Centroid Longitude",
]

unique_calls = data.groupby(group_cols).size().reset_index(name="group_n")

# create group id for each unique call
unique_calls["group_id"] = unique_calls.index

# merge group id back into full dataset for later matching purposes
data = data.merge(unique_calls[group_cols + ["group_id"]], on=group_cols, how="left")

print(f"Length of full dataset: {len(data)}")
print(f"Number of unique API calls: {len(unique_calls)}")

Length of full dataset: 13007182
Number of unique API calls: 2782549


### Formatting for API call
Convert day types into specific representative dates, with separate year, month, and day columns with data as strings.

In [ ]:
unique_calls["representative_year"] = "2026"
unique_calls["representative_month"] = "04"
unique_calls["representative_day"] = data["day_type"].map({0: 22, 1: 25, 2: 26})

## Random Sample

In [ ]:
# Sample 2000 calls, weighting likelihood of being sampled by group size
api_2000 = unique_calls.sample(2000, weights="group_n", random_state=888)

# Split into 4 csvs
molly_500 = api_2000[0:500]
sarah_500 = api_2000[500:1000]
sabrina_500 = api_2000[1000:1500]
waleed_500 = api_2000[1500:2000]

molly_500.to_csv("./data/molly_500.csv", index=False)
sarah_500.to_csv("./data/sarah_500.csv", index=False)
sabrina_500.to_csv("./data/sabrina_500.csv", index=False)
waleed_500.to_csv("./data/waleed_500.csv", index=False)

In [ ]:
# update dataset to avoid repeat pulls
data_for_api_10k = unique_calls[~unique_calls["group_id"].isin(api_2000["group_id"])]

# Sample 10000 calls, weighting likelihood of being sampled by group size
api_10k = data_for_api_10k.sample(
    40000, weights="group_n", replace=True, random_state=11
)

# Split into 4 csvs
molly_10k = api_10k[0:10000]
sarah_10k = api_10k[10000:20000]
sabrina_10k = api_10k[20000:30000]
waleed_10k = api_10k[30000:40000]

molly_10k.to_csv("./data/molly_10k.csv", index=False)
sarah_10k.to_csv("./data/sarah_10k.csv", index=False)
sabrina_10k.to_csv("./data/sabrina_10k.csv", index=False)
waleed_10k.to_csv("./data/waleed_10k.csv", index=False)

In [ ]:
# update dataset to avoid repeat pulls
data_for_api_58k = data_for_api_10k[
    ~data_for_api_10k["group_id"].isin(api_10k["group_id"])
]

# Sample 10000 calls, weighting likelihood of being sampled by group size
api_58k = data_for_api_58k.sample(
    58000, weights="group_n", replace=True, random_state=222
)

# Split into 4 csvs
molly_58k = api_58k[0:14500]
sarah_58k = api_58k[14500 : (14500 * 2)]
sabrina_58k = api_58k[(14500 * 2) : (14500 * 3)]
waleed_58k = api_58k[(14500 * 3) : 58000]

molly_58k.to_csv("./data/molly_58k.csv", index=False)
sarah_58k.to_csv("./data/sarah_58k.csv", index=False)
sabrina_58k.to_csv("./data/sabrina_58k.csv", index=False)
waleed_58k.to_csv("./data/waleed_58k.csv", index=False)

### Aggregate data by unique ride
Aggregate rides by unique ride group, taking average of distance, cost, time, and number of rides in that ride group, and a representative value for every other dimension

Author: Sabrina

In [ ]:
# `data` as is defined under the grouping data section
# Trip length as an int
data["Int Trip Seconds"] = data['Trip Seconds'].str.replace(",", "").astype(int)

# Turn all money into floats
col_name = [
    "Fare", "Tip", "Additional Charges", "Trip Total"
]
for col in col_name:
    data[f"Float {col}"] = data[col].str[1:].astype(float)

,mean
group_id,
0,1196.875000
1,639.000000
2,486.000000
3,698.000000
4,392.333333
...,...
2782544,266.000000
2782545,553.000000
2782546,277.000000


In [52]:
ride_groups = data.groupby('group_id').aggregate({
    'Int Trip Seconds': 'mean',
    'Trip Miles': 'mean',
    "Float Fare": "mean",
    "Float Tip": "mean",
    "Float Additional Charges": "mean",
    "Float Trip Total": "mean",
    "Pickup Census Tract": "first",
    "Dropoff Census Tract": "first",
    "Pickup Centroid Latitude": "first",
    "Pickup Centroid Longitude": "first",
    "Dropoff Centroid Latitude": "first",
    "Dropoff Centroid Longitude": "first",
    "start_hour": "first",
    "day_type": "first",
    "Trip ID": "count"
    })

In [59]:
ride_groups = ride_groups.rename(columns={
    'Int Trip Seconds': 'Average Trip Seconds',
    'Trip Miles': 'Average Miles',
    "Float Fare": "Average Fare",
    "Float Tip": "Average Tip",
    "Float Additional Charges": "Average Additional Charges",
    "Float Trip Total": "Average Trip Total",
    "Trip ID": "Count"
    })

In [ ]:
print(f"Agregated data has {len(ride_groups)} rows")
ride_groups.to_csv("./data/ride_groups.csv", index=False)
print("Sucessfully wrote ride_group dataset to data/ride_groups.csv")